In [2]:
import pandas as pd
df = pd.read_csv('tech_layoff_2026.csv')
df.head(2)

,company,layoff_date,jobs_cut,pct_workforce_cut,sector,country,hq_city,ai_cited,reason_stated,company_revenue_2025_bn,...,layoffs_2024,layoffs_2025,verified_source,month,quarter,region,layoff_size_category,stock_reaction,laid_off_vs_headcount_pct,data_as_of
0,Amazon,2026-01-15,16000,2.7,E-Commerce/Cloud,USA,Seattle,False,Reduce bureaucracy and management layers,716.9,...,4000,14000,CNBC / NetworkWorld,January 2026,Q1 2026,North America,Mega (5K+),Positive,1.03,"March 18, 2026"
1,Block,2026-02-28,4000,40.0,Fintech,USA,San Francisco,True,AI tools replace roles enabling smaller teams,22.4,...,0,1000,CNBC / Crunchbase,February 2026,Q1 2026,North America,Large (2K-5K),Positive,40.00,"March 18, 2026"


In [3]:
df['stock_reaction'].value_counts()

stock_reaction
Positive    17
Negative     8
Neutral      3
Name: count, dtype: int64

In [4]:
import numpy as np
numeric_df = df.select_dtypes(include=np.number)
numeric_column  = numeric_df.columns.to_list()
print("before feature engineering numeric column: ")
numeric_column

before feature engineering numeric column: 


['jobs_cut',
 'pct_workforce_cut',
 'company_revenue_2025_bn',
 'pre_layoff_headcount',
 'stock_change_day_pct',
 'simultaneous_ai_investment_bn',
 'layoffs_2024',
 'layoffs_2025',
 'laid_off_vs_headcount_pct']

# Feature Engineering

In [ ]:
# df['revenue_per_emp'] = df["company_revenue_2025_bn"] * 1e9 / df["pre_layoff_headcount"]
df['revenue_per_emp'] = df["company_revenue_2025_bn"] / df["pre_layoff_headcount"]

df["ai_investment_ratio"] = (
    df["simultaneous_ai_investment_bn"] / (df["company_revenue_2025_bn"] + 1e-9)
)

df["layoff_acceleration"] = df["layoffs_2025"] - df["layoffs_2024"]

df["cut_scale_normalized"] = df["jobs_cut"] / (df["pre_layoff_headcount"] + 1)


In [6]:
numeric_df = df.select_dtypes(include=np.number)
numeric_column  = numeric_df.columns.to_list()
print("after feature engineering numeric columns: ")
numeric_column

after feature engineering numeric columns: 


['jobs_cut',
 'pct_workforce_cut',
 'company_revenue_2025_bn',
 'pre_layoff_headcount',
 'stock_change_day_pct',
 'simultaneous_ai_investment_bn',
 'layoffs_2024',
 'layoffs_2025',
 'laid_off_vs_headcount_pct',
 'revenue_per_emp',
 'ai_investment_ratio',
 'layoff_acceleration',
 'cut_scale_normalized']

In [7]:
df["ai_cited_flag"] = df["ai_cited"].astype(int)

In [8]:
df.drop(columns = ['quarter'] , inplace = True)

In [9]:
category = df.select_dtypes(include='object')
category.columns.tolist()

['company',
 'layoff_date',
 'sector',
 'country',
 'hq_city',
 'reason_stated',
 'roles_most_affected',
 'replacement_roles',
 'ceo_quote',
 'verified_source',
 'month',
 'region',
 'layoff_size_category',
 'stock_reaction',
 'data_as_of']

In [10]:
df.drop(columns = ['verified_source','hq_city','ceo_quote'],inplace = True)

In [11]:
df.drop(columns = ['layoff_date','reason_stated','roles_most_affected','replacement_roles','data_as_of'] , inplace=True)

In [12]:
df.drop(columns = ['month'] , inplace=True)

In [13]:
df.drop(columns = ['company'] , inplace=True)

In [14]:
df['sector'].value_counts()

sector
Telecommunications          3
Semiconductors              2
Enterprise Software         2
Manufacturing               2
Social Media/VR             1
Fintech                     1
E-Commerce/Cloud            1
E-Commerce                  1
Design Software             1
CRM/SaaS                    1
Social Media                1
Logistics Software          1
Networking/Cybersecurity    1
EV Batteries                1
Grocery Tech                1
Retail Pharmacy             1
AI Research                 1
Interior Design Tech        1
Cybersecurity               1
Automotive Tech             1
Enterprise SaaS             1
Insurance                   1
Social Media/AI             1
Name: count, dtype: int64

In [15]:
sector_map = {
    "E-Commerce/Cloud"        : "Cloud/Platform",
    "E-Commerce"              : "Cloud/Platform",
    "AI Research"             : "AI/Data",
    "Social Media/AI"         : "AI/Data",
    "Fintech"                 : "Fintech/SaaS",
    "CRM/SaaS"                : "Fintech/SaaS",
    "Enterprise Software"     : "Fintech/SaaS",
    "Enterprise SaaS"         : "Fintech/SaaS",
    "Design Software"         : "Fintech/SaaS",
    "Logistics Software"      : "Fintech/SaaS",
    "Interior Design Tech"    : "Fintech/SaaS",
    "Social Media/VR"         : "Consumer Tech",
    "Social Media"            : "Consumer Tech",
    "Retail Pharmacy"         : "Consumer Tech",
    "Grocery Tech"            : "Consumer Tech",
    "Networking/Cybersecurity": "Infra/Hardware",
    "Cybersecurity"           : "Infra/Hardware",
    "Semiconductors"          : "Infra/Hardware",
    "Telecommunications"      : "Infra/Hardware",
    "Automotive Tech"         : "Industrial/Other",
    "EV Batteries"            : "Industrial/Other",
    "Manufacturing"           : "Industrial/Other",
    "Insurance"               : "Industrial/Other",
}
df["sector_group"] = df["sector"].map(sector_map).fillna("Other")


In [16]:
df.drop(columns = ['sector'] , inplace = True)

In [23]:
df.drop(columns = ['country'] , inplace = True)

In [24]:
df.columns

Index(['jobs_cut', 'pct_workforce_cut', 'ai_cited', 'company_revenue_2025_bn',
       'pre_layoff_headcount', 'stock_change_day_pct',
       'simultaneous_ai_investment_bn', 'layoffs_2024', 'layoffs_2025',
       'region', 'layoff_size_category', 'stock_reaction',
       'laid_off_vs_headcount_pct', 'revenue_per_emp', 'ai_investment_ratio',
       'layoff_acceleration', 'cut_scale_normalized', 'ai_cited_flag',
       'sector_group'],
      dtype='object')

In [25]:
df.drop(columns = ['laid_off_vs_headcount_pct','stock_change_day_pct','ai_cited'] , inplace=True)

In [27]:
df.head(2)

,jobs_cut,pct_workforce_cut,company_revenue_2025_bn,pre_layoff_headcount,simultaneous_ai_investment_bn,layoffs_2024,layoffs_2025,region,layoff_size_category,stock_reaction,revenue_per_emp,ai_investment_ratio,layoff_acceleration,cut_scale_normalized,ai_cited_flag,sector_group
0,16000,2.7,716.9,1550000,100.0,4000,14000,North America,Mega (5K+),Positive,0.000463,0.139489,10000,0.010323,0,Cloud/Platform
1,4000,40.0,22.4,10000,2.0,0,1000,North America,Large (2K-5K),Positive,0.002240,0.089286,1000,0.399960,1,Fintech/SaaS


In [28]:
df.to_csv('ml_train_data.csv',index=False)